# ECS708 Coursework — Fashion-MNIST
## Part 0 — Setup (0.1)

In [3]:
!pip install numpy pandas scikit-learn matplotlib torch torchvision

  Using cached torch-2.15.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (36 kB)
  Using cached torchvision-0.29.2-py3-none-macosx_14_0_arm64.whl.metadata (5.7 kB)
  Using cached filelock-4.2.10-py3-none-any.whl.metadata (2.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.6/138.6 MB 816.8 kB/s eta 0:00:0000:0100:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 638.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 931.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 738.2 kB/s eta 0:00:00:--:--

[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, ru

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split
from torchvision.datasets import FashionMNIST

SEED = 42
tr = FashionMNIST("data", train=True, download=True)
te = FashionMNIST("data", train=False, download=True)
y_all, y_test = tr.targets.numpy(), te.targets.numpy()

idx = np.arange(len(y_all))
train_idx, val_idx = train_test_split(idx, test_size=10_000, stratify=y_all, random_state=SEED)

def strat_subset(pool, labels, n, seed):
    sub, _ = train_test_split(pool, train_size=n, stratify=labels[pool], random_state=seed)
    return np.sort(sub)

cpu_train_idx = strat_subset(train_idx, y_all, 10_000, SEED)
clu_train_idx = strat_subset(train_idx, y_all, 3_000, SEED)
clu_val_idx   = strat_subset(val_idx,   y_all, 1_000, SEED)
clu_test_idx  = strat_subset(np.arange(len(y_test)), y_test, 1_000, SEED)

np.savez("splits/splits_seed42.npz", seed=SEED, train=np.sort(train_idx), val=np.sort(val_idx),
         cpu_train=cpu_train_idx, clu_train=clu_train_idx, clu_val=clu_val_idx, clu_test=clu_test_idx)
print("saved")

saved


In [13]:
s = np.load("splits/splits_seed42.npz")
for name in ["train", "val", "cpu_train", "clu_train", "clu_val", "clu_test"]:
    labels = (y_test if name == "clu_test" else y_all)[s[name]]
    print(f"{name:10} {len(labels):6}  {np.bincount(labels)}")

train       50000  [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]
val         10000  [1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]
cpu_train   10000  [1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]
clu_train    3000  [300 300 300 300 300 300 300 300 300 300]
clu_val      1000  [100 100 100 100 100 100 100 100 100 100]
clu_test     1000  [100 100 100 100 100 100 100 100 100 100]


In [14]:
import pandas as pd, os
cols = ["id","task","question","hypothesis","changed_variable","fixed_conditions",
        "split","seed","config","metric","result","output_path","notes"]
if not os.path.exists("results/experiment_register.csv"):
    pd.DataFrame(columns=cols).to_csv("results/experiment_register.csv", index=False)
print(pd.read_csv("results/experiment_register.csv"))

Empty DataFrame
Columns: [id, task, question, hypothesis, changed_variable, fixed_conditions, split, seed, config, metric, result, output_path, notes]
Index: []
